# EEC 289A — Representation Analysis: Phonetic Geometry in SMT Space

This notebook evaluates the **Sparse Manifold Transform (SMT)** embedding as a semantic distance metric for speech patches, fulfilling the representation analysis goals from the project proposal:

1. **Load LibriSpeech** with forced-alignment phoneme labels (via `torchaudio` MMS_FA pipeline)
2. **Build SMT representation** on speech patches (ZCA → K-means → GEVD spectral basis → FISTA embedding)
3. **kNN phoneme classification** — compare accuracy in raw MFCC space vs ZCA-whitened space vs SMT embedding space
4. **Cluster geometry** — quantify phoneme cluster separability (silhouette score, intra/inter-class distance ratio, Davies-Bouldin index) in each space
5. **Coarticulation** — examine whether phoneme-boundary transitions are geometrically consistent (low intra-speaker variance) in SMT space

> **Reference:** Chen et al., *Sparse Manifold Transform* (SMT); Lu et al., *Scaling Non-Parametric Sampling with Representation* (arXiv 2510.22196)

## Section 1 — Imports & Global Configuration

In [ ]:
import sys, os, random, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torchaudio
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import soundfile as sf

from tqdm import tqdm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    silhouette_score, davies_bouldin_score,
)
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from scipy.linalg import eigh
from scipy.sparse import csr_matrix
from scipy.spatial.distance import cdist

warnings.filterwarnings("ignore")

# --- Global constants ----------------------------------------------------------
SR_TARGET   = 16_000          # resample all LibriSpeech audio to 16 kHz
HOP_MS      = 10.0            # hop length in ms
WIN_MS      = 25.0            # analysis window in ms
N_MFCC      = 40              # MFCC dimensionality
PATCH_MS    = 300.0           # temporal patch width (ms)
N_CLUSTERS  = 128             # K-means clusters for SMT basis
SMT_DIM     = 32              # spectral embedding dimension
LAMBDA_FISTA= 0.1             # FISTA sparsity coefficient
MIN_PH_PATCHES = 10           # minimum patches per phoneme class to include

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"torchaudio {torchaudio.__version__} | librosa {librosa.__version__}")
PROJ_DIR = Path("/home/angela/Code/eec-289/Final-Proj")
DATA_DIR = PROJ_DIR / "librispeech_data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

## Section 2 — SMT Core Functions (ZCA · K-means · Spectral Decomp · FISTA)

These mirror the functions in `SMT.py` and `Final.ipynb`, re-imported here for self-containment.

In [ ]:
# ── Preprocessing ────────────────────────────────────────────────────────────

def extract_mfcc_patches(audio: np.ndarray, sr: int, patch_ms: float = PATCH_MS,
                          hop_ms: float = HOP_MS, win_ms: float = WIN_MS,
                          n_mfcc: int = N_MFCC) -> tuple[np.ndarray, int, int]:
    """
    Extract sliding-window MFCC patches from a waveform.

    Returns:
        patches  : (n_patches, n_mfcc, patch_frames)
        hop_len  : samples per hop
        win_len  : samples per window
    """
    hop_len = max(1, int(round(sr * hop_ms / 1000)))
    win_len = max(hop_len, int(round(sr * win_ms / 1000)))
    n_fft   = win_len
    n_mels  = max(n_mfcc * 2, 80)

    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_fft=n_fft, win_length=win_len,
        hop_length=hop_len, n_mels=n_mels, power=2.0,
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    mfcc    = librosa.feature.mfcc(S=log_mel, n_mfcc=n_mfcc)  # (n_mfcc, T)

    patch_frames = max(1, int(round(patch_ms / 1000 * sr / hop_len)))
    n_frames = mfcc.shape[1]
    if n_frames < patch_frames:
        return np.empty((0, n_mfcc, patch_frames)), hop_len, win_len

    starts  = range(0, n_frames - patch_frames + 1)
    patches = np.stack([mfcc[:, s:s + patch_frames] for s in starts])
    return patches, hop_len, win_len


def zca_whitening_matrix(X: np.ndarray) -> np.ndarray:
    """X : (features, observations) → ZCA matrix (features, features)"""
    sigma = np.cov(X, rowvar=True)
    U, S, _ = np.linalg.svd(sigma)
    return U @ np.diag(1.0 / np.sqrt(S + 1e-5)) @ U.T


def preprocess_patches(patches_flat: np.ndarray):
    """Mean-center, ZCA-whiten, L2-normalize.  Input: (N, D). Returns whitened (N, D)."""
    mean = patches_flat.mean(axis=0)
    centered = patches_flat - mean
    zca  = zca_whitening_matrix(centered.T)
    wht  = (zca @ centered.T).T
    wht  = normalize(wht, norm='l2')
    return wht, mean, zca


# ── K-means + spectral decomposition ─────────────────────────────────────────

def fit_kmeans(patches: np.ndarray, n_clusters: int) -> MiniBatchKMeans:
    km = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    km.fit(patches)
    return km


def spectral_decomp(labels: np.ndarray, utterance_bounds: list, n_clusters: int):
    """Solve GEVD: Mu=λVu with slowness matrix M and whitening constraint V."""
    n_samples = labels.shape[0]
    A = csr_matrix(
        (np.ones(n_samples, dtype=np.float32), (np.arange(n_samples), labels)),
        shape=(n_samples, n_clusters),
    )
    M = np.zeros((n_clusters, n_clusters))
    for (start, end) in utterance_bounds:
        for j in tqdm(range(start, end - 1), desc="  spectral decomp", leave=False):
            if j + 1 < A.shape[0]:
                diff = (A[j + 1] - A[j]).T
                M += diff @ diff.T

    A_T = A.T
    V = (A_T @ A_T.T) / n_samples + 1e-6 * np.eye(n_clusters)
    eigvals, eigvecs = eigh(M, V)
    return eigvals, eigvecs


def build_smt_basis(patches_flat: np.ndarray, labels: np.ndarray,
                    utterance_bounds: list, n_clusters: int, smt_dim: int,
                    kmeans: MiniBatchKMeans) -> np.ndarray:
    """
    Build (feature_dim, smt_dim) spectral basis matrix.
    Columns = lifted spectral eigenvectors from K-means cluster centres.
    """
    _, eigvecs = spectral_decomp(labels, utterance_bounds, n_clusters)
    d   = min(smt_dim, eigvecs.shape[1] - 1)
    P   = eigvecs[:, 1:d + 1].T   # (d, n_clusters) — skip trivial eigenvec
    cc  = normalize(kmeans.cluster_centers_, axis=1, norm='l2')
    basis = cc.T @ P.T             # (feature_dim, d)
    basis = normalize(basis, axis=0, norm='l2')
    return basis


# ── FISTA sparse coding ───────────────────────────────────────────────────────

def fista_sparse_code(I: torch.Tensor, basis: torch.Tensor,
                      lambd: float = 0.1, n_iter: int = 50) -> torch.Tensor:
    """
    Positive FISTA sparse coding:  min_{z≥0} 0.5||I - B z||² + λ||z||₁
    I     : (feature_dim, N)
    basis : (feature_dim, d)
    Returns z : (d, N)
    """
    d    = basis.shape[1]
    N    = I.shape[1]
    BtI  = basis.t() @ I               # (d, N)
    BtB  = basis.t() @ basis           # (d, d)
    L    = torch.linalg.norm(BtB, ord=2).item() + 1e-6
    z    = torch.zeros(d, N, device=I.device)
    y    = z.clone()
    t    = 1.0
    for _ in range(n_iter):
        grad  = BtB @ y - BtI
        zn    = torch.clamp(y - grad / L - lambd / L, min=0.0)
        t_new = (1 + (1 + 4 * t ** 2) ** 0.5) / 2
        y     = zn + (t - 1) / t_new * (zn - z)
        z, t  = zn, t_new
    return z


def embed_patches(patches_flat_np: np.ndarray, basis_np: np.ndarray,
                  device: torch.device) -> np.ndarray:
    """Embed (N, D) patches into SMT space → (N, smt_dim)."""
    pf  = torch.tensor(patches_flat_np, dtype=torch.float32, device=device).T
    bas = torch.tensor(basis_np, dtype=torch.float32, device=device)
    z   = fista_sparse_code(pf, bas, lambd=LAMBDA_FISTA, n_iter=50)
    return z.T.cpu().numpy()   # (N, smt_dim)


print("SMT functions defined.")

## Section 3 — Load LibriSpeech + Phoneme Forced Alignment

`torchaudio.pipelines.MMS_FA` is a pretrained CTC model whose token vocabulary is phonemes.  
We use `torchaudio.functional.forced_align` to obtain frame-accurate phoneme boundaries at 20 ms resolution.

> **First run:** downloads `dev-clean` (~340 MB) and the MMS_FA model (~1.1 GB) once, then caches both.  
> Set `N_UTTERANCES` to a smaller number (e.g. 50) to iterate faster.

In [ ]:
N_UTTERANCES = 200    # Number of utterances to process (increase for production)

# ── Load MMS_FA pipeline ──────────────────────────────────────────────────────
print("Loading MMS_FA forced alignment model (downloads ~1.1 GB on first run)...")
FA_BUNDLE  = torchaudio.pipelines.MMS_FA
FA_MODEL   = FA_BUNDLE.get_model().to(device)
FA_LABELS  = FA_BUNDLE.get_labels(star=None)   # phoneme vocabulary (no <star>)
# MMS_FA expects 16 kHz audio
FA_SR      = FA_BUNDLE.sample_rate
print(f"MMS_FA vocab size: {len(FA_LABELS)}  sample_rate: {FA_SR} Hz")

# ── Load LibriSpeech dev-clean ────────────────────────────────────────────────
print("\nLoading LibriSpeech dev-clean (downloads ~340 MB on first run)...")
librispeech = torchaudio.datasets.LIBRISPEECH(
    root=str(DATA_DIR), url="dev-clean", download=True,
)
print(f"Total utterances: {len(librispeech)}")

In [ ]:
def align_utterance(waveform: torch.Tensor, sr: int, transcript: str,
                    model, labels, target_sr: int = FA_SR):
    """
    Run forced alignment on one utterance.

    Returns list of (phoneme_str, start_frame, end_frame) at the model's
    output frame rate (~20 ms per frame for MMS_FA).
    """
    # Resample to model's target sample rate
    if sr != target_sr:
        waveform = torchaudio.functional.resample(waveform, sr, target_sr)

    # Clean transcript → character sequence expected by MMS_FA
    text = transcript.strip().lower()
    # MMS_FA uses character tokens including space
    # Build token ids
    label2id = {l: i for i, l in enumerate(labels)}
    # Convert text to token ID sequence (space becomes "|" in MMS_FA vocab)
    SPACE = "|"
    tokens = []
    for ch in text:
        if ch == " ":
            token_ch = SPACE
        elif ch == "'":
            token_ch = "'"
        else:
            token_ch = ch
        if token_ch in label2id:
            tokens.append(label2id[token_ch])

    if not tokens:
        return []

    with torch.inference_mode():
        emission, _ = model(waveform.to(device))   # (1, T_frames, vocab)

    # torchaudio forced align
    targets = torch.tensor([tokens], dtype=torch.int32, device=device)
    aligned = torchaudio.functional.forced_align(
        emission, targets, blank=0
    )
    # aligned.tokens: (1, T_frames) token ids at each output frame
    # Collapse repeated tokens + blanks to get spans
    spans = []
    token_seq = aligned.tokens[0].cpu().tolist()
    n_frames  = len(token_seq)

    prev_tok, span_start = -1, 0
    for fi, tok in enumerate(token_seq):
        if tok != prev_tok:
            if prev_tok > 0 and prev_tok < len(labels):
                spans.append((labels[prev_tok], span_start, fi))
            span_start = fi
            prev_tok   = tok
    if prev_tok > 0 and prev_tok < len(labels):
        spans.append((labels[prev_tok], span_start, n_frames))

    return spans   # list of (char, start_frame, end_frame)


print("align_utterance defined.")

In [ ]:
# ── Map characters → broad phoneme classes (English) ─────────────────────────
# MMS_FA aligns at the CHARACTER level.  We map to 6 broad acoustic classes
# that represent meaningfully distinct phoneme categories for the evaluation.

PHONEME_MAP = {
    # Class: vowels (open / close, tense / lax)
    **{c: "vowel"      for c in "aeiou"},
    # Class: stops (oral plosives)
    **{c: "stop"       for c in "pbtdkg"},
    # Class: fricatives
    **{c: "fricative"  for c in "fvszh"},
    # Class: nasals
    **{c: "nasal"      for c in "mn"},
    # Class: liquids
    **{c: "liquid"     for c in "lr"},
    # Class: glides / approximants
    **{c: "glide"      for c in "wyj"},
    # Consonant clusters / other
    **{c: "other"      for c in "cqx"},
}

def char_to_phoneme_class(char: str) -> str:
    return PHONEME_MAP.get(char.lower(), "other")


# ── Extract phoneme-labeled MFCC patches ──────────────────────────────────────

def process_utterance(waveform: torch.Tensor, sr: int, transcript: str,
                      model, labels,
                      n_mfcc: int = N_MFCC, patch_ms: float = PATCH_MS,
                      hop_ms: float = HOP_MS, win_ms: float = WIN_MS):
    """
    1. Run forced alignment on one utterance.
    2. Extract MFCC patches and assign each patch's label from the
       dominant phoneme class at its center frame.

    Returns:
        patches  : (n_patches, n_mfcc * patch_frames)  flattened patch vectors
        ph_labels: (n_patches,)  phoneme class string per patch
        hop_len  : hop length in samples (for time calculations)
    """
    FA_HOP_MS = 20.0   # MMS_FA output is ~20 ms per frame
    wav_np    = waveform.squeeze(0).numpy()

    # Resample waveform to MMS_FA sample rate for alignment
    if sr != FA_SR:
        wf_fa = torchaudio.functional.resample(waveform, sr, FA_SR)
    else:
        wf_fa = waveform

    # --- forced alignment -------------------------------------------------------
    spans = align_utterance(wf_fa, FA_SR, transcript, model, labels, FA_SR)
    if not spans:
        return None, None, None

    # Convert alignment spans (in FA frames) → character labels at each MFCC frame
    hop_len = max(1, int(round(sr * hop_ms / 1000)))
    n_mfcc_frames = waveform.shape[-1] // hop_len + 1
    fa_frames_per_mfcc = (FA_HOP_MS / hop_ms)    # ratio of frame rates

    frame_labels = ["other"] * n_mfcc_frames
    for (char, fa_start, fa_end) in spans:
        ph_class = char_to_phoneme_class(char)
        mfcc_start = int(fa_start  * fa_frames_per_mfcc)
        mfcc_end   = int(fa_end    * fa_frames_per_mfcc)
        for f in range(max(0, mfcc_start), min(n_mfcc_frames, mfcc_end)):
            frame_labels[f] = ph_class

    # --- extract patches and inherit center-frame label ------------------------
    patches_3d, hop_len, _ = extract_mfcc_patches(
        wav_np, sr, patch_ms=patch_ms, hop_ms=hop_ms,
        win_ms=win_ms, n_mfcc=n_mfcc,
    )
    if patches_3d.shape[0] == 0:
        return None, None, None

    patch_frames = patches_3d.shape[2]
    pad_left = patch_frames // 2

    ph_labels = []
    for i in range(patches_3d.shape[0]):
        center = min(i + pad_left, len(frame_labels) - 1)
        ph_labels.append(frame_labels[center])

    patches_flat = patches_3d.reshape(patches_3d.shape[0], -1)
    return patches_flat, np.array(ph_labels), hop_len


print("process_utterance defined.")

In [ ]:
# ── Collect phoneme-labeled patches from LibriSpeech ─────────────────────────

all_patches  = []    # list of (n_patches, D) arrays
all_labels   = []    # list of (n_patches,) string arrays
# utterance bounds (start, end) in global patch index — needed for spectral decomp
utterance_bounds = []
global_idx = 0
n_failed   = 0

indices = list(range(min(N_UTTERANCES, len(librispeech))))
random.shuffle(indices)

for i in tqdm(indices, desc="Aligning utterances"):
    wf, sr_utt, transcript, *_ = librispeech[i]
    try:
        patches_flat, ph_labels, _ = process_utterance(
            wf, sr_utt, transcript, FA_MODEL, FA_LABELS)
    except Exception as e:
        n_failed += 1
        continue

    if patches_flat is None or patches_flat.shape[0] < 2:
        n_failed += 1
        continue

    n = patches_flat.shape[0]
    all_patches.append(patches_flat)
    all_labels.append(ph_labels)
    utterance_bounds.append((global_idx, global_idx + n))
    global_idx += n

all_patches_np = np.concatenate(all_patches, axis=0).astype(np.float32)
all_labels_np  = np.concatenate(all_labels,  axis=0)

print(f"\nCollected {all_patches_np.shape[0]:,} patches from "
      f"{len(utterance_bounds)} utterances  ({n_failed} failed).")
print(f"Feature dim per patch: {all_patches_np.shape[1]}")

# --- Class distribution -------------------------------------------------------
from collections import Counter
counts = Counter(all_labels_np)
print("\nPhoneme class distribution:")
for cls, cnt in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {cls:12s}  {cnt:6d}  ({100*cnt/len(all_labels_np):.1f}%)")

## Section 4 — Build SMT Representation on Speech Patches

Pipeline: raw MFCC patches → ZCA whitening → K-means → GEVD spectral basis → SMT embedding via FISTA sparse coding.

In [ ]:
# ── 1. ZCA whitening + L2 normalisation ──────────────────────────────────────
print("ZCA whitening...")
wht_patches, patch_mean, zca_mat = preprocess_patches(all_patches_np)

# ── 2. K-means on whitened patches ───────────────────────────────────────────
n_clusters = min(N_CLUSTERS, wht_patches.shape[0] // 4)
print(f"K-means (k={n_clusters})...")
kmeans = fit_kmeans(wht_patches, n_clusters=n_clusters)
labels_km = kmeans.labels_

# ── 3. Spectral decomposition: GEVD  Mu = λVu ────────────────────────────────
print("Spectral decomposition (GEVD)...")
smt_basis_np = build_smt_basis(
    wht_patches, labels_km, utterance_bounds,
    n_clusters, SMT_DIM, kmeans,
)
print(f"SMT basis shape: {smt_basis_np.shape}  (feature_dim × {SMT_DIM})")

# ── 4. Embed all patches into SMT space (FISTA) ───────────────────────────────
print("Embedding patches into SMT space (FISTA sparse coding)...")
# Apply same ZCA + L2 normalisation to patches before embedding
#   (they are already stored in wht_patches)
smt_embeddings = embed_patches(wht_patches, smt_basis_np, device)
print(f"SMT embeddings shape: {smt_embeddings.shape}")

# ── Filter out 'other' class and very rare phoneme classes ───────────────────
counts = Counter(all_labels_np)
valid_classes = {cls for cls, cnt in counts.items()
                 if cnt >= MIN_PH_PATCHES and cls != "other"}
mask = np.array([l in valid_classes for l in all_labels_np])

X_raw  = all_patches_np[mask]              # (n, feature_dim)  — raw MFCC
X_zca  = wht_patches[mask]                 # (n, feature_dim)  — ZCA whitened
X_smt  = smt_embeddings[mask]              # (n, smt_dim)       — SMT embedded
y      = all_labels_np[mask]               # (n,)  phoneme class labels

print(f"\nKept {mask.sum():,} patches across {len(valid_classes)} phoneme classes: "
      f"{sorted(valid_classes)}")

## Section 5 — kNN Phoneme Classification: MFCC vs ZCA vs SMT

We evaluate how well k-nearest neighbours can recover phoneme class labels in each space.  
A higher cross-validated accuracy indicates that the space better separates phoneme categories.

In [ ]:
K_NEIGHBORS = 5    # k for kNN
N_SPLITS    = 5    # stratified k-fold CV folds

# Sub-sample for speed if dataset is very large
MAX_SAMPLES = 5000
if len(y) > MAX_SAMPLES:
    rng = np.random.default_rng(42)
    idx = rng.choice(len(y), MAX_SAMPLES, replace=False)
    X_raw_s, X_zca_s, X_smt_s, y_s = (
        X_raw[idx], X_zca[idx], X_smt[idx], y[idx])
else:
    X_raw_s, X_zca_s, X_smt_s, y_s = X_raw, X_zca, X_smt, y

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
knn = KNeighborsClassifier(n_neighbors=K_NEIGHBORS, metric="euclidean", n_jobs=-1)

results = {}
for name, X in [("Raw MFCC", X_raw_s), ("ZCA-whitened MFCC", X_zca_s), ("SMT embedding", X_smt_s)]:
    scores = cross_val_score(knn, X, y_s, cv=cv, scoring="accuracy")
    results[name] = scores
    print(f"{name:22s}: {scores.mean():.3f} ± {scores.std():.3f}  "
          f"(per-fold: {np.round(scores, 3)})")

# ── Bar chart ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
names  = list(results.keys())
means  = [v.mean() for v in results.values()]
stds   = [v.std()  for v in results.values()]
colors = ["#4c72b0", "#dd8452", "#55a868"]
bars   = ax.bar(names, means, yerr=stds, capsize=6, color=colors, alpha=0.85)
ax.set_ylabel("k-NN Accuracy (5-fold CV)")
ax.set_title(f"Phoneme Classification Accuracy  (k={K_NEIGHBORS})")
ax.set_ylim(0, min(1.0, max(means) * 1.4))
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, m + 0.01,
            f"{m:.3f}", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.savefig(PROJ_DIR / "knn_accuracy_comparison.png", dpi=150)
plt.show()
print("Saved knn_accuracy_comparison.png")

In [ ]:
# ── Confusion matrix for best-performing space ────────────────────────────────
best_name = max(results, key=lambda k: results[k].mean())
best_X    = {"Raw MFCC": X_raw_s, "ZCA-whitened MFCC": X_zca_s,
             "SMT embedding": X_smt_s}[best_name]

# Single train/test split for plotting
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(
    best_X, y_s, test_size=0.25, random_state=42, stratify=y_s)
knn.fit(X_tr, y_tr)
y_pred = knn.predict(X_te)

classes_sorted = sorted(valid_classes)
cm = confusion_matrix(y_te, y_pred, labels=classes_sorted, normalize="true")

fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes_sorted)
disp.plot(ax=ax, cmap="Blues", colorbar=True, values_format=".2f")
ax.set_title(f"Normalised Confusion Matrix — {best_name}")
plt.tight_layout()
plt.savefig(PROJ_DIR / "confusion_matrix_best.png", dpi=150)
plt.show()
print(f"Best space: {best_name}")

## Section 6 — Phoneme Cluster Geometry: Euclidean vs SMT Space

We quantify phoneme cluster separability using three complementary metrics:

| Metric | Better when… |
|---|---|
| **Silhouette score** $s(i) = \frac{b(i) - a(i)}{\max(a(i),\, b(i))}$ | Closer to **+1** |
| **Davies-Bouldin index** $DB = \frac{1}{k}\sum_i \max_{j\neq i}\frac{s_i + s_j}{d_{ij}}$ | Closer to **0** |
| **Inter/Intra distance ratio** $\rho = \bar{d}_{\text{inter}} / \bar{d}_{\text{intra}}$ | **Larger** is better |

In [ ]:
N_GEOM = min(3000, len(y))   # subsample for tractable computation
rng    = np.random.default_rng(0)
gi     = rng.choice(len(y), N_GEOM, replace=False)
y_g    = y[gi]

# Encode string labels to integers for sklearn metrics
label_enc = {cls: k for k, cls in enumerate(sorted(valid_classes))}
y_int = np.array([label_enc[l] for l in y_g])


def geometry_metrics(X: np.ndarray, y_int: np.ndarray, y_str: np.ndarray,
                     name: str) -> dict:
    """Compute silhouette, DB index, and inter/intra distance ratio."""
    sil = silhouette_score(X, y_int, metric="euclidean", sample_size=min(2000, len(y_int)),
                           random_state=42)
    db  = davies_bouldin_score(X, y_int)

    # Per-class centroids → inter-class distances
    classes = sorted(set(y_str))
    centroids = np.vstack([X[y_str == c].mean(axis=0) for c in classes])
    inter_d = cdist(centroids, centroids, metric="euclidean")
    np.fill_diagonal(inter_d, np.nan)
    mean_inter = np.nanmean(inter_d)

    # Intra-class: mean distance from each point to its class centroid
    intra_ds = []
    for c, centroid in zip(classes, centroids):
        pts = X[y_str == c]
        intra_ds.append(np.linalg.norm(pts - centroid, axis=1).mean())
    mean_intra = np.mean(intra_ds)
    ratio = mean_inter / (mean_intra + 1e-9)

    print(f"  {name:22s} | silhouette={sil:+.4f}  DB={db:.4f}  inter/intra={ratio:.3f}")
    return {"silhouette": sil, "db": db, "ratio": ratio,
            "mean_inter": mean_inter, "mean_intra": mean_intra}


print("Cluster geometry metrics:")
geom = {}
for space_name, X_space in [
    ("Raw MFCC",          X_raw[gi]),
    ("ZCA-whitened MFCC", X_zca[gi]),
    ("SMT embedding",     X_smt[gi]),
]:
    geom[space_name] = geometry_metrics(X_space, y_int, y_g, space_name)

# ── Grouped bar chart ─────────────────────────────────────────────────────────
space_names = list(geom.keys())
metrics_to_plot = {
    "Silhouette ↑": [geom[s]["silhouette"] for s in space_names],
    "DB index ↓ (neg.)" : [-geom[s]["db"]       for s in space_names],
    "Inter/Intra ↑ (log)": [np.log1p(geom[s]["ratio"]) for s in space_names],
}
x = np.arange(len(space_names))
width = 0.25
colors = ["#4c72b0", "#dd8452", "#55a868"]

fig, ax = plt.subplots(figsize=(9, 5))
for k, (metric, vals) in enumerate(metrics_to_plot.items()):
    ax.bar(x + k * width, vals, width, label=metric, color=colors[k], alpha=0.85)
ax.set_xticks(x + width)
ax.set_xticklabels(space_names)
ax.legend(loc="lower right")
ax.set_title("Phoneme Cluster Geometry: Raw MFCC vs ZCA vs SMT")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
plt.tight_layout()
plt.savefig(PROJ_DIR / "cluster_geometry_comparison.png", dpi=150)
plt.show()
print("Saved cluster_geometry_comparison.png")

In [ ]:
# ── PCA projection (2D) of each space ─────────────────────────────────────────
N_VIZ = min(1500, len(y_g))
vi    = rng.choice(len(y_g), N_VIZ, replace=False)
classes_sorted = sorted(valid_classes)
cmap  = cm.get_cmap("tab10", len(classes_sorted))
cdict = {cls: cmap(i) for i, cls in enumerate(classes_sorted)}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (space_name, X_space) in zip(axes, [
        ("Raw MFCC",          X_raw[gi][vi]),
        ("ZCA-whitened MFCC", X_zca[gi][vi]),
        ("SMT embedding",     X_smt[gi][vi]),
]):
    pca    = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(X_space)
    for cls in classes_sorted:
        mask_c = y_g[vi] == cls
        ax.scatter(coords[mask_c, 0], coords[mask_c, 1],
                   c=[cdict[cls]], alpha=0.35, s=12, label=cls)
    ax.set_title(f"{space_name}\n(PCA PC1 vs PC2,  var: "
                 f"{pca.explained_variance_ratio_[:2].sum()*100:.1f}%)")
    ax.legend(markerscale=2, fontsize=8, loc="best")
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")

plt.suptitle("PCA Phoneme Cluster Visualisation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(PROJ_DIR / "pca_phoneme_clusters.png", dpi=150)
plt.show()
print("Saved pca_phoneme_clusters.png")

## Section 7 — Coarticulation Trajectory Analysis in SMT Space

The key question: **does the SMT manifold linearly interpolate between adjacent phoneme classes?**

For each pair of adjacent phoneme classes in an utterance, we extract the sequence of SMT-embedded patches spanning the boundary and ask whether:
1. Trajectories are **directional** (cross the boundary in consistent order in PCA space)
2. **Intra-transition variance** is low (all instances of the same A→B pair follow a similar path)

A low Dynamic Time Warping (DTW) distance between same-pair instances would confirm coarticulation consistency.

In [ ]:
BOUNDARY_WINDOW = 15    # frames before & after the transition to capture

def simple_dtw(a: np.ndarray, b: np.ndarray) -> float:
    """Simple O(n*m) DTW between two 2D trajectories (T, d)."""
    n, m = len(a), len(b)
    D = np.full((n + 1, m + 1), np.inf)
    D[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = np.linalg.norm(a[i - 1] - b[j - 1])
            D[i, j] = cost + min(D[i-1, j], D[i, j-1], D[i-1, j-1])
    return float(D[n, m])


# ── Build per-utterance phoneme label sequences ───────────────────────────────
# We need sequential labels, so we build them from utterance_bounds & all_labels_np

transitions: dict[tuple, list] = defaultdict(list)  # (A,B) → list of trajectory arrays

global_start = 0
for start, end in utterance_bounds:
    utt_labels  = all_labels_np[start:end]
    utt_smt     = smt_embeddings[start:end]     # (n_patches, smt_dim)

    for t in range(1, len(utt_labels)):
        prev_cls = utt_labels[t - 1]
        curr_cls = utt_labels[t]
        if prev_cls == curr_cls or prev_cls == "other" or curr_cls == "other":
            continue
        if prev_cls not in valid_classes or curr_cls not in valid_classes:
            continue

        # Extract a window of SMT patches around the transition
        win_start = max(0, t - BOUNDARY_WINDOW)
        win_end   = min(len(utt_smt), t + BOUNDARY_WINDOW)
        traj = utt_smt[win_start:win_end]   # (window_len, smt_dim)
        if len(traj) < 4:
            continue
        transitions[(prev_cls, curr_cls)].append(traj)

print(f"Found {sum(len(v) for v in transitions.values()):,} transition instances "
      f"across {len(transitions)} unique phoneme pairs.")

# ── Report intra-pair DTW consistency (top-5 most frequent pairs) ─────────────
sorted_pairs = sorted(transitions.items(), key=lambda x: -len(x[1]))
print("\nTop transition pairs — mean pairwise DTW distance (lower = more consistent):")
dtw_results = {}
for pair, trajs in sorted_pairs[:8]:
    if len(trajs) < 2:
        continue
    # Truncate all trajectories to same length (shortest in group)
    min_len = min(len(t) for t in trajs)
    trajs_t = [t[:min_len] for t in trajs]
    # Sample up to 10 for speed
    sample  = trajs_t[:10]
    dtw_ds  = []
    for ii in range(len(sample)):
        for jj in range(ii + 1, len(sample)):
            dtw_ds.append(simple_dtw(sample[ii], sample[jj]))
    mean_dtw = np.mean(dtw_ds) if dtw_ds else float("nan")
    dtw_results[pair] = mean_dtw
    print(f"  {pair[0]:10s} → {pair[1]:10s}  "
          f"({len(trajs):4d} instances)  mean DTW={mean_dtw:.3f}")

In [ ]:
# ── Trajectory PCA visualisation — top 4 phoneme transition pairs ─────────────
n_show = min(4, len(sorted_pairs))
fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 4), sharey=False)
if n_show == 1:
    axes = [axes]

pca_2d = PCA(n_components=2, random_state=42)
pca_2d.fit(X_smt)    # fit PCA on ALL SMT patches for a global reference frame

for ax, (pair, trajs) in zip(axes, sorted_pairs[:n_show]):
    min_len = min(len(t) for t in trajs)
    trajs_t = [t[:min_len] for t in trajs[:20]]   # up to 20 trajectories

    colormap = cm.get_cmap("viridis", len(trajs_t))
    for ti, traj in enumerate(trajs_t):
        proj = pca_2d.transform(traj)   # (T, 2)
        ax.plot(proj[:, 0], proj[:, 1], alpha=0.55, linewidth=1.2,
                color=colormap(ti))
        # Mark start (circle) and end (star)
        ax.scatter(*proj[0], marker="o", s=35, color=colormap(ti), zorder=5)
        ax.scatter(*proj[-1], marker="*", s=60, color=colormap(ti), zorder=5)

    ax.set_title(f"/{pair[0]}/ → /{pair[1]}/\n({len(trajs)} instances)")
    ax.set_xlabel("SMT PC 1"); ax.set_ylabel("SMT PC 2")
    # Vertical dashed line approximately at transition midpoint
    ax.axvline(0, color="gray", linestyle="--", linewidth=0.7, alpha=0.6)

plt.suptitle("Coarticulation Trajectories in SMT Space\n(circle=onset, star=offset)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(PROJ_DIR / "coarticulation_trajectories.png", dpi=150)
plt.show()
print("Saved coarticulation_trajectories.png")

## Section 8 — Temporal Co-occurrence in SMT Space

We compute a **phoneme transition matrix** (bigram co-occurrence) from the LibriSpeech utterances.  
The matrix captures which phoneme classes most often follow each other — a proxy for the statistical structure that non-parametric sampling must model.

In [ ]:
classes_sorted = sorted(valid_classes)
cls_to_idx     = {c: i for i, c in enumerate(classes_sorted)}
n_cls          = len(classes_sorted)

# ── Build bigram co-occurrence matrix ─────────────────────────────────────────
cooc = np.zeros((n_cls, n_cls), dtype=np.float32)

for start, end in utterance_bounds:
    utt_labels = all_labels_np[start:end]
    for t in range(1, len(utt_labels)):
        a, b = utt_labels[t - 1], utt_labels[t]
        if a in cls_to_idx and b in cls_to_idx:
            cooc[cls_to_idx[a], cls_to_idx[b]] += 1.0

# Row-normalise → transition probabilities
row_sums = cooc.sum(axis=1, keepdims=True)
cooc_prob = cooc / np.maximum(row_sums, 1.0)

# ── Heatmap ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im0 = axes[0].imshow(cooc,      cmap="YlOrRd")
axes[0].set_title("Bigram Co-occurrence (raw counts)")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(cooc_prob, cmap="YlOrRd", vmin=0, vmax=1)
axes[1].set_title("Transition Probability P(B|A)")
plt.colorbar(im1, ax=axes[1])

for axi in axes:
    axi.set_xticks(range(n_cls)); axi.set_xticklabels(classes_sorted, rotation=45, ha="right")
    axi.set_yticks(range(n_cls)); axi.set_yticklabels(classes_sorted)
    axi.set_xlabel("Next phoneme class (B)")
    axi.set_ylabel("Current phoneme class (A)")

plt.suptitle("Phoneme Temporal Co-occurrence — LibriSpeech dev-clean",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(PROJ_DIR / "phoneme_cooccurrence.png", dpi=150)
plt.show()
print("Saved phoneme_cooccurrence.png")

# ── SMT-space centroid distance vs. co-occurrence frequency ───────────────────
# Do phonemes that frequently co-occur have closer SMT centroids?
centroids = np.vstack([
    X_smt[y == c].mean(axis=0)
    for c in classes_sorted
    if (y == c).any()
])
centroid_dists = cdist(centroids, centroids, metric="euclidean")
np.fill_diagonal(centroid_dists, np.nan)

# Flatten upper-triangle only
triu_idx = np.triu_indices(n_cls, k=1)
cooc_vals = cooc[triu_idx] + cooc[triu_idx[1], triu_idx[0]]   # symmetric sum
dist_vals  = centroid_dists[triu_idx]
mask_valid_pair = ~np.isnan(dist_vals)

corr = np.corrcoef(cooc_vals[mask_valid_pair], dist_vals[mask_valid_pair])[0, 1]

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(cooc_vals[mask_valid_pair], dist_vals[mask_valid_pair], alpha=0.7)
ax.set_xlabel("Co-occurrence count (A↔B)")
ax.set_ylabel("SMT centroid distance")
ax.set_title(f"Co-occurrence vs SMT Distance\nPearson r = {corr:.3f}")
plt.tight_layout()
plt.savefig(PROJ_DIR / "cooc_vs_smt_dist.png", dpi=150)
plt.show()
print(f"Pearson correlation (co-occurrence ↔ centroid distance): r = {corr:.4f}")
print("Negative r would indicate that frequent co-occurrence → close in SMT space (expected).")

## Section 9 — Summary & Interpretation

| Goal | Cell | Metric |
|---|---|---|
| kNN phoneme classification (MFCC vs ZCA vs SMT) | §5 | Cross-validated accuracy |
| Phoneme cluster geometry | §6 | Silhouette, DB index, inter/intra ratio |
| PCA visualisation of cluster separability | §6 | Visual |
| Coarticulation trajectory consistency | §7 | Mean pairwise DTW |
| Temporal co-occurrence structure | §8 | Bigram transition matrix, Pearson r vs. SMT distance |

> **Expected finding**: If SMT captures meaningful semantic structure in audio, we expect:
> - **Higher kNN accuracy** in SMT space vs raw MFCC (better phoneme separability)
> - **Higher silhouette / lower DB** in SMT space (tighter, more separated phoneme clusters)
> - **Low DTW variance** for same A→B phoneme transitions across utterances (consistent coarticulation geometry)
> - **Negative correlation** between co-occurrence frequency and SMT centroid distance (frequent co-occurrences → nearby clusters)